## Downloading Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s5e7', output_dir='datasets/predict_introverts_from_extroverts')

print("Path to competition files:", path)

/home/theju-bot/Documents/ML Engineering/Projects/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████████████████████████████████████████████████████████████████████| 191k/191k [00:00<00:00, 324kB/s]

Extracting files...
Path to competition files: datasets/predict_introverts_from_extroverts


## Load & explore the data set 

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path('datasets/predict_introverts_from_extroverts')

train_df = pd.read_csv(data_dir/'train.csv')
test = pd.read_csv(data_dir/'test.csv')
sample_submission = pd.read_csv(data_dir/'sample_submission.csv')

In [2]:
train_df.head()

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert


In [3]:
train_df.shape

(18524, 9)

In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18524 entries, 0 to 18523
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         18524 non-null  int64  
 1   Time_spent_Alone           17334 non-null  float64
 2   Stage_fear                 16631 non-null  str    
 3   Social_event_attendance    17344 non-null  float64
 4   Going_outside              17058 non-null  float64
 5   Drained_after_socializing  17375 non-null  str    
 6   Friends_circle_size        17470 non-null  float64
 7   Post_frequency             17260 non-null  float64
 8   Personality                18524 non-null  str    
dtypes: float64(5), int64(1), str(3)
memory usage: 1.3 MB


In [5]:
train_df.describe()

,id,Time_spent_Alone,Social_event_attendance,Going_outside,Friends_circle_size,Post_frequency
count,18524.000000,17334.000000,17344.000000,17058.000000,17470.000000,17260.000000
mean,9261.500000,3.137764,5.265106,4.044319,7.996737,4.982097
std,5347.562529,3.003786,2.753359,2.062580,4.223484,2.879139
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4630.750000,1.000000,3.000000,3.000000,5.000000,3.000000
50%,9261.500000,2.000000,5.000000,4.000000,8.000000,5.000000
75%,13892.250000,4.000000,8.000000,6.000000,12.000000,7.000000
max,18523.000000,11.000000,10.000000,7.000000,15.000000,10.000000


In [6]:
train_df.isnull().sum()

id                              0
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
Personality                     0
dtype: int64

In [7]:
(train_df.isnull().mean() * 100).sort_values(ascending=False)

Stage_fear                   10.219175
Going_outside                 7.914057
Post_frequency                6.823580
Time_spent_Alone              6.424098
Social_event_attendance       6.370114
Drained_after_socializing     6.202764
Friends_circle_size           5.689916
id                            0.000000
Personality                   0.000000
dtype: float64

In [8]:
train_df.nunique().sort_values()

Stage_fear                       2
Drained_after_socializing        2
Personality                      2
Going_outside                    8
Post_frequency                  11
Social_event_attendance         11
Time_spent_Alone                12
Friends_circle_size             16
id                           18524
dtype: int64

## Data Processing

In [9]:
X = train_df.drop(columns=['id', 'Personality'])
y = train_df['Personality']

In [23]:
y.value_counts(normalize=True) * 100

Personality
Extrovert    73.95271
Introvert    26.04729
Name: proportion, dtype: float64

In [36]:
cat_col = X.select_dtypes(include=['str']).columns

for col in X[cat_col].columns:
    print(f'{(X[col].value_counts(normalize=True, dropna=False) * 100).round(2).astype('str') + '%'}\n')


Stage_fear
No     68.07%
Yes    21.71%
NaN    10.22%
Name: proportion, dtype: str

Drained_after_socializing
No     71.87%
Yes    21.93%
NaN      6.2%
Name: proportion, dtype: str



In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
X_train.nunique().sort_values()

Stage_fear                    2
Drained_after_socializing     2
Going_outside                 8
Social_event_attendance      11
Post_frequency               11
Time_spent_Alone             12
Friends_circle_size          16
dtype: int64

In [25]:
X_train.isnull().sum()

Time_spent_Alone              981
Stage_fear                   1510
Social_event_attendance       930
Going_outside                1180
Drained_after_socializing     906
Friends_circle_size           848
Post_frequency                988
dtype: int64

In [24]:
for col in X_train[X_train.columns].columns:
    print(f'{(X_train[col].value_counts(normalize=True, dropna=False) * 100).round(2).astype('str') + '%'}\n')


Time_spent_Alone
0.0     16.84%
2.0     16.57%
3.0     16.53%
1.0     16.06%
NaN      6.62%
4.0      5.83%
5.0      3.38%
8.0      3.19%
6.0      3.12%
10.0     3.12%
9.0      3.11%
7.0      3.07%
11.0     2.57%
Name: proportion, dtype: str

Stage_fear
No     68.14%
Yes    21.67%
NaN    10.19%
Name: proportion, dtype: str

Social_event_attendance
4.0     11.27%
9.0     11.15%
7.0     10.97%
5.0     10.76%
6.0     10.74%
8.0      10.4%
3.0      9.22%
NaN      6.28%
2.0      5.77%
0.0      5.75%
1.0      5.07%
10.0     2.62%
Name: proportion, dtype: str

Going_outside
5.0    15.89%
3.0     15.2%
6.0    14.68%
4.0    14.63%
7.0    11.57%
NaN     7.96%
0.0     7.07%
2.0     6.78%
1.0     6.22%
Name: proportion, dtype: str

Drained_after_socializing
No     72.02%
Yes    21.86%
NaN     6.11%
Name: proportion, dtype: str

Friends_circle_size
5.0     8.88%
4.0     7.19%
12.0    7.15%
8.0     7.11%
10.0    6.83%
11.0    6.67%
14.0    6.41%
6.0     6.18%
7.0     6.13%
NaN     5.72%
9.0     5.65%

In [29]:
X_train['Stage_fear'] = X_train['Stage_fear'].fillna("Unknown")
X_train['Drained_after_socializing'] = X_train['Drained_after_socializing'].fillna("Unknown")

## Training Models

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path('datasets/predict_introverts_from_extroverts')

train_df = pd.read_csv(data_dir/'train.csv')
test = pd.read_csv(data_dir/'test.csv')
sample_submission = pd.read_csv(data_dir/'sample_submission.csv')

In [3]:
X = train_df.drop(columns=['id', 'Personality'])
y = train_df['Personality']

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
from sklearn.impute import SimpleImputer

cat_cols = ['Stage_fear', 'Drained_after_socializing']

imputer = SimpleImputer(
    strategy='constant',
    fill_value='Unknown'
)

test_features = test.drop(columns=['id']).copy()

X_train[cat_cols] = imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = imputer.transform(X_test[cat_cols])
test_features[cat_cols] = imputer.transform(test_features[cat_cols])
test[cat_cols] = imputer.transform(test[cat_cols])

### Random Forest

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(include='str').columns

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])


rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

In [6]:
%%time

rf_model.fit(X_train, y_train)

CPU times: user 1.69 s, sys: 77.4 ms, total: 1.76 s
Wall time: 413 ms


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['Extrovert','Introvert']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Time_spent_Alone','Stage_fear','Social_event_attendance',..., 'Drained_after_socializing','Friends_circle_size','Post_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are d

In [7]:
rf_model.score(X_test, y_test)

0.9678812415654521

In [48]:
predictions = rf_model.predict(test)
predictions

array(['Extrovert', 'Introvert', 'Extrovert', ..., 'Extrovert',
       'Extrovert', 'Introvert'], shape=(6175,), dtype=object)

### XGBoost

In [36]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [38]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

cat_cols = X_train.select_dtypes(include='str').columns

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_pipeline, cat_cols)
    ],
    remainder='passthrough'
)

xgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

In [12]:
%%time

xgb_model.fit(X_train, y_train_encoded)

CPU times: user 2.25 s, sys: 291 ms, total: 2.54 s
Wall time: 501 ms


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Time_spent_Alone','Stage_fear','Social_event_attendance',..., 'Drained_after_socializing','Friends_circle_size','Post_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By 

In [14]:
xgb_model.score(X_test, y_test_encoded)

0.970310391363023

In [15]:
y_pred_encoded = xgb_model.predict(test)
y_pred_encoded

array([0, 1, 0, ..., 0, 0, 1], shape=(6175,))

In [16]:
predictions = label_encoder.inverse_transform(y_pred_encoded.astype(int))

### Cat Boost

In [6]:
from catboost import CatBoostClassifier

cat_cols = X_train.select_dtypes(include='str').columns.tolist()

cat_model = CatBoostClassifier(
    random_seed=42,
    thread_count=-1,
    verbose=False, 
    auto_class_weights='Balanced',
    allow_writing_files=False,
)

In [7]:
%%time

cat_model.fit(X_train, y_train, cat_features=cat_cols)

CPU times: user 1min 32s, sys: 24 s, total: 1min 56s
Wall time: 8.64 s


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', random_seed=42, verbose=False)

In [8]:
cat_model.score(X_test, y_test)

np.float64(0.9711201079622133)

In [16]:
predictions = cat_model.predict(test_features)
predictions

array(['Extrovert', 'Introvert', 'Extrovert', ..., 'Extrovert',
       'Extrovert', 'Introvert'], shape=(6175,), dtype=object)

### LightGBM

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from lightgbm import LGBMClassifier

cat_cols = X_train.select_dtypes(include='str').columns

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_pipeline, cat_cols)
    ],
    remainder='passthrough'
)

lgbm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

In [6]:
%%time

lgbm_model.fit(X_train, y_train)

CPU times: user 4.7 s, sys: 11.8 ms, total: 4.72 s
Wall time: 349 ms


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['Extrovert','Introvert']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Time_spent_Alone','Stage_fear','Social_event_attendance',..., 'Drained_after_socializing','Friends_circle_size','Post_frequency']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defau

In [7]:
lgbm_model.score(X_test, y_test)

0.9713900134952766

In [9]:
predictions = lgbm_model.predict(test)
predictions

array(['Extrovert', 'Introvert', 'Extrovert', ..., 'Extrovert',
       'Extrovert', 'Introvert'], shape=(6175,), dtype=object)

## Hyperparameter Tuning

### Catboost

In [16]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    verbose=0,
    random_seed=42,
    allow_writing_files=False
)

params = {
    'depth': [4, 5, 6, 7, 8, 9, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'iterations': [500, 800, 1000, 1500],
    'l2_leaf_reg': [1, 3, 5, 7, 10],
    'random_strength': [0, 1, 2, 5],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=30,
    scoring='accuracy',
    cv=skf,
    n_jobs=1,
    random_state=42,
    verbose=1
)

In [17]:
%%time

search.fit(X_train, y_train, cat_features=cat_cols)

print("Best parameters:", search.best_params_)
print("Best CV score:", search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'random_strength': 2, 'learning_rate': 0.01, 'l2_leaf_reg': 7, 'iterations': 500, 'depth': 9}
Best CV score: 0.9683515039057964
CPU times: user 5h 13s, sys: 2h 4min 4s, total: 7h 4min 18s
Wall time: 30min 52s


In [18]:
params = {'random_strength': 2, 'learning_rate': 0.01, 'l2_leaf_reg': 7, 'iterations': 500, 'depth': 9}

In [23]:
%%time

cat_model_2 = CatBoostClassifier(random_seed=42, thread_count=-1, verbose=False, allow_writing_files=False, **params)

cat_model_2.fit(X_train, y_train, cat_features=cat_cols)

CPU times: user 1min 29s, sys: 42.2 s, total: 2min 11s
Wall time: 9.81 s


CatBoostClassifier(allow_writing_files=False, depth=9, iterations=500, l2_leaf_reg=7, learning_rate=0.01, random_seed=42, random_strength=2, verbose=False)

In [29]:
%%time

eval_model(params)

Fold 1 Accuracy: 0.9666
Fold 2 Accuracy: 0.9642
Fold 3 Accuracy: 0.9720
Fold 4 Accuracy: 0.9696
Fold 5 Accuracy: 0.9693

Mean Accuracy: 0.968352
Std: 0.0027
CPU times: user 6min 23s, sys: 3min 17s, total: 9min 41s
Wall time: 42.5 s


In [25]:
predictions = cat_model_2.predict(test_features)
predictions

array(['Extrovert', 'Introvert', 'Extrovert', ..., 'Extrovert',
       'Extrovert', 'Introvert'], shape=(6175,), dtype=object)

### XGBoost

In [39]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

params = {
    'model__n_estimators': [200, 300, 500, 800, 1000],
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5, 7, 10],
    'model__gamma': [0, 0.1, 0.3, 0.5, 1],
    'model__reg_alpha': [0, 0.01, 0.1, 1, 10],
    'model__reg_lambda': [1, 3, 5, 10, 20]
}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=params,
    n_iter=30,
    scoring='accuracy',
    cv=skf,
    n_jobs=1,
    random_state=42,
    verbose=1
)

In [42]:
print("Best Parameters:")
print(search.best_params_)

print(f"\nBest CV Accuracy: {search.best_score_:.4f}")

Best Parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 10, 'model__reg_alpha': 10, 'model__n_estimators': 200, 'model__min_child_weight': 3, 'model__max_depth': 3, 'model__learning_rate': 0.2, 'model__gamma': 0.3, 'model__colsample_bytree': 1.0}

Best CV Accuracy: 0.9686


In [45]:
params_pipeline = {
    'model__subsample': 0.8, 
    'model__reg_lambda': 10, 
    'model__reg_alpha': 10, 
    'model__n_estimators': 200, 
    'model__min_child_weight': 3, 
    'model__max_depth': 3, 
    'model__learning_rate': 0.2, 
    'model__gamma': 0.3, 
    'model__colsample_bytree': 1.0
}

In [48]:
best_xgb = search.best_estimator_

In [61]:
y_pred_encoded = best_xgb.predict(test)
y_pred_encoded

array([0, 1, 0, ..., 0, 0, 1], shape=(6175,))

In [62]:
predictions = label_encoder.inverse_transform(y_pred_encoded.astype(int))

## Submission

In [63]:
ids = test['id']

submission = pd.DataFrame({
    'id': ids,
    'is_fraud': predictions
})

In [64]:
submission

,id,is_fraud
0,18524,Extrovert
1,18525,Introvert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Introvert
...,...,...
6170,24694,Extrovert
6171,24695,Introvert
6172,24696,Extrovert
6173,24697,Extrovert


In [66]:
submission.to_csv('datasets/predict_introverts_from_extroverts/submissions/submission_6_XGBClassifier_with_hypr.csv', index=False)

## Saving and getting the model

In [23]:
import joblib
from pathlib import Path

model_dir = Path('./models')
model_dir.mkdir(parents=True, exist_ok=True)

feature_cols = X_train.columns.tolist()

artifacts = {
    'xgb_classifier': xgb_model_1,

    'ohe_encoder': ohe_encoder,

    'scaler': scaler,

    'num_cols': num_cols.tolist(),
    'cat_cols': cat_cols.tolist(),
    'bool_cols': bool_cols.tolist(),
    
    'feature_cols': feature_cols,

    'X_test': X_test,
    'y_test': y_test,
    'X_train': X_train,
    'y_train': y_train,
    'test': test,
}

artifact_path = model_dir / 'predict_introverts_from_extroverts.joblib'

joblib.dump(artifacts, artifact_path)

print(f"Artifacts saved to: {artifact_path}")


Artifacts saved to: models/predicting_credit_card_fraud.joblib


In [3]:
import joblib

artifacts = joblib.load('./models/predict_introverts_from_extroverts.joblib')

best_xgb = artifacts['best_xgb'] 

In [51]:
artifacts['best_xgb'] = best_xgb

In [53]:
import joblib

joblib.dump(artifacts, './models/predict_introverts_from_extroverts.joblib')

['./models/predict_introverts_from_extroverts.joblib']

## Eval Model

In [44]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

def eval_model(name, params):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    
        X_fit = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
    
        y_fit = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]
    
        model = name(**params)
    
        model.fit(X_fit, y_fit, cat_features=cat_cols)
    
        y_pred = model.predict(X_val)
    
        score = accuracy_score(y_val, y_pred)
        scores.append(score)
    
        print(f"Fold {fold} Accuracy: {score:.4f}")
    
    print(f"\nMean Accuracy: {np.mean(scores):.6f}")
    print(f"Std: {np.std(scores):.4f}")